In [3]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path
from urllib.parse import urlencode

import requests


BASE_URL = "https://datahub.uniandes.edu.co"
DATASET_DOI = "doi:10.57924/BLUILW"  # Cambia esto si luego usas otro dataset
OUTPUT_DIR = Path("data/raw/elca_2016")
TIMEOUT = 60


def safe_filename(name: str) -> str:
    """
    Limpia nombres de archivos para evitar problemas con caracteres especiales.
    """
    name = name.strip().replace("/", "_").replace("\\", "_")
    name = re.sub(r"[^\w\-.() ]", "_", name, flags=re.UNICODE)
    return name


def get_dataset_metadata(base_url: str, dataset_doi: str) -> dict:
    """
    Consulta los metadatos de un dataset en Dataverse usando su DOI.
    """
    url = f"{base_url}/api/datasets/:persistentId/?{urlencode({'persistentId': dataset_doi})}"
    response = requests.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    payload = response.json()

    if payload.get("status") != "OK":
        raise RuntimeError(f"No se pudo obtener el dataset: {payload}")

    return payload["data"]


def extract_files_from_metadata(metadata: dict) -> list[dict]:
    """
    Extrae la lista de archivos disponibles dentro de los metadatos del dataset.
    """
    latest_version = metadata.get("latestVersion", {})
    files = latest_version.get("files", [])

    extracted = []
    for f in files:
        data_file = f.get("dataFile", {})
        extracted.append(
            {
                "id": data_file.get("id"),
                "filename": data_file.get("filename"),
                "description": f.get("description"),
                "contentType": data_file.get("contentType"),
                "filesize": data_file.get("filesize"),
                "md5": data_file.get("md5"),
                "persistentId": data_file.get("persistentId"),
            }
        )
    return extracted


def download_file_by_id(base_url: str, file_id: int, destination: Path) -> None:
    """
    Descarga un archivo individual usando el file id de Dataverse.
    """
    url = f"{base_url}/api/access/datafile/{file_id}"
    with requests.get(url, stream=True, timeout=TIMEOUT) as response:
        response.raise_for_status()
        with open(destination, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    out_file.write(chunk)


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Consultando metadatos del dataset: {DATASET_DOI}")
    metadata = get_dataset_metadata(BASE_URL, DATASET_DOI)

    metadata_path = OUTPUT_DIR / "dataset_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    files = extract_files_from_metadata(metadata)
    if not files:
        print("No se encontraron archivos en el dataset.")
        return

    manifest_path = OUTPUT_DIR / "files_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(files, f, ensure_ascii=False, indent=2)

    print(f"Se encontraron {len(files)} archivos.")
    for i, file_info in enumerate(files, start=1):
        file_id = file_info["id"]
        filename = safe_filename(file_info["filename"] or f"file_{file_id}")
        destination = OUTPUT_DIR / filename

        if destination.exists():
            print(f"[{i}/{len(files)}] Ya existe, se omite: {filename}")
            continue

        print(f"[{i}/{len(files)}] Descargando: {filename}")
        try:
            download_file_by_id(BASE_URL, file_id, destination)
        except Exception as e:
            print(f"Error descargando {filename}: {e}")

    print("Proceso terminado.")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"Error general: {exc}")
        sys.exit(1)

Error general: [Errno 30] Read-only file system: 'data'


SystemExit: 1

/Applications/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
